In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score


In [2]:
val_binary = pd.read_csv("Extraction/val_binary.tsv", sep="\t")
test_binary = pd.read_csv("Extraction/test_binary.tsv", sep="\t")
metadata_cols = ["Entry", "Length", "Sequence"]
concept_cols = [c for c in val_binary.columns if c not in metadata_cols]

In [5]:
def calculate_f1_array(precision, recall):
    denom = precision + recall
    return np.divide(
        2 * precision * recall,
        denom,
        out=np.zeros_like(denom, dtype=float),
        where=denom > 0
    )

def compare_features_to_concepts_fast(
    A,
    binary_df,
    concept_cols,
    thresholds=(0, 0.15, 0.5, 0.6, 0.8),
):
    """
    A: normalized activations, shape [n_proteins, n_features]
    binary_df: val_binary/test_binary
    concept_cols: concept label columns

    Returns dataframe with:
    concept, feature, threshold, precision, recall, f1
    """

    A = np.asarray(A)
    Y = binary_df[concept_cols].values.astype(bool)

    n_proteins, n_features = A.shape
    n_concepts = Y.shape[1]

    positives = Y.sum(axis=0)  # [n_concepts]
    results = []

    for threshold in thresholds:
        A_bin = A > threshold  # [n_proteins, n_features]

        # Matrix multiplication gives TP:
        # Y.T: [n_concepts, n_proteins]
        # A_bin: [n_proteins, n_features]
        # tp: [n_concepts, n_features]
        tp = Y.T.astype(np.int32) @ A_bin.astype(np.int32)

        pred_pos = A_bin.sum(axis=0)  # [n_features]
        fp = pred_pos[None, :] - tp

        precision = np.divide(
            tp,
            tp + fp,
            out=np.zeros_like(tp, dtype=float),
            where=(tp + fp) > 0,
        )

        recall = np.divide(
            tp,
            positives[:, None],
            out=np.zeros_like(tp, dtype=float),
            where=positives[:, None] > 0,
        )

        f1 = calculate_f1_array(precision, recall)

        concept_idx, feature_idx = np.nonzero(tp > 0)

        df_t = pd.DataFrame({
            "concept": [concept_cols[i] for i in concept_idx],
            "feature": feature_idx,
            "threshold": threshold,
            "precision": precision[concept_idx, feature_idx],
            "recall": recall[concept_idx, feature_idx],
            "f1": f1[concept_idx, feature_idx],
            #"tp": tp[concept_idx, feature_idx],
            #"fp": fp[concept_idx, feature_idx],
            #"positive_labels": positives[concept_idx],
        })

        results.append(df_t)

    return pd.concat(results, ignore_index=True)

cls

In [7]:
val_data = np.load(
    "Extraction/val_features/embeddings_cls_8/embeddings_cls_sparse_pca_8_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_cls_8/embeddings_cls_sparse_pca_8_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 8)
(25000, 8)
val f1: 0.0673
test f1: 0.06466
Validation pairs with F1 > 0.5: 1
Those also with test F1 > 0.5: 1
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0005634,5,0.6,0.49179,0.671741,0.56785,0.49422,0.679897,0.572377


In [8]:
val_data = np.load(
    "Extraction/val_features/embeddings_cls_16/embeddings_cls_sparse_pca_16_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_cls_16/embeddings_cls_sparse_pca_16_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 16)
(25000, 16)
val f1: 0.07218
test f1: 0.06823
Validation pairs with F1 > 0.5: 1
Those also with test F1 > 0.5: 1
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0005634,5,0.5,0.411919,0.852114,0.555369,0.414207,0.852793,0.55759


In [9]:
val_data = np.load(
    "Extraction/val_features/embeddings_cls_32/embeddings_cls_sparse_pca_32_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_cls_32/embeddings_cls_sparse_pca_32_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 32)
(25000, 32)
val f1: 0.08617
test f1: 0.08074
Validation pairs with F1 > 0.5: 1
Those also with test F1 > 0.5: 1
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0005634,5,0.5,0.407928,0.856055,0.552553,0.411408,0.858774,0.556309


In [10]:
val_data = np.load(
    "Extraction/val_features/embeddings_cls_64/embeddings_cls_sparse_pca_64_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_cls_64/embeddings_cls_sparse_pca_64_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 64)
(25000, 64)
val f1: 0.08711
test f1: 0.08072
Validation pairs with F1 > 0.5: 1
Those also with test F1 > 0.5: 1
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0005634,1,0.5,0.380996,0.84858,0.525881,0.384131,0.854832,0.530069


In [11]:
val_data = np.load(
    "Extraction/val_features/embeddings_cls_128/embeddings_cls_sparse_pca_128_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_cls_128/embeddings_cls_sparse_pca_128_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 128)
(25000, 128)
val f1: 0.09789
test f1: 0.09037
Validation pairs with F1 > 0.5: 1
Those also with test F1 > 0.5: 1
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0005634,1,0.5,0.372756,0.855104,0.519188,0.377242,0.866114,0.525569


In [12]:
val_data = np.load(
    "Extraction/val_features/embeddings_cls_320/embeddings_cls_sparse_pca_320_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_cls_320/embeddings_cls_sparse_pca_320_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 320)
(25000, 320)
val f1: 0.09799
test f1: 0.08663
Validation pairs with F1 > 0.5: 2
Those also with test F1 > 0.5: 2
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0005634,1,0.5,0.376225,0.918037,0.533723,0.377913,0.919125,0.535604
1,GO:0005737,1,0.5,0.354278,0.866013,0.502846,0.359638,0.866083,0.508234


layer_mean

In [13]:
val_data = np.load(
    "Extraction/val_features/embeddings_layer_mean_8/embeddings_layer_mean_sparse_pca_8_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_layer_mean_8/embeddings_layer_mean_sparse_pca_8_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 8)
(25000, 8)
val f1: 0.07081
test f1: 0.06816
Validation pairs with F1 > 0.5: 0
Those also with test F1 > 0.5: 0
Survival rate: 0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1


In [14]:
val_data = np.load(
    "Extraction/val_features/embeddings_layer_mean_16/embeddings_layer_mean_sparse_pca_16_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_layer_mean_16/embeddings_layer_mean_sparse_pca_16_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 16)
(25000, 16)
val f1: 0.09056
test f1: 0.0866
Validation pairs with F1 > 0.5: 3
Those also with test F1 > 0.5: 2
Survival rate: 0.6666666666666666


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0007186,11,0.6,0.546917,0.488038,0.515803,0.547945,0.473934,0.508259
1,transmembrane,1,0.5,0.999502,0.345070,0.513023,0.999493,0.338019,0.505188


In [15]:
val_data = np.load(
    "Extraction/val_features/embeddings_layer_mean_32/embeddings_layer_mean_sparse_pca_32_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_layer_mean_32/embeddings_layer_mean_sparse_pca_32_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 32)
(25000, 32)
val f1: 0.10535
test f1: 0.10014
Validation pairs with F1 > 0.5: 4
Those also with test F1 > 0.5: 2
Survival rate: 0.5


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,transmembrane,1,0.5,0.930261,0.398660,0.558134,0.937396,0.384985,0.545808
1,GO:0007186,11,0.6,0.549865,0.488038,0.517110,0.553719,0.476303,0.512102


In [16]:
val_data = np.load(
    "Extraction/val_features/embeddings_layer_mean_64/embeddings_layer_mean_sparse_pca_64_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_layer_mean_64/embeddings_layer_mean_sparse_pca_64_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 64)
(25000, 64)
val f1: 0.11794
test f1: 0.11135
Validation pairs with F1 > 0.5: 7
Those also with test F1 > 0.5: 6
Survival rate: 0.8571428571428571


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0005506,33,0.8,0.976526,0.506083,0.666667,0.973404,0.491935,0.653571
1,heme,33,0.8,0.962441,0.488095,0.647709,0.941489,0.460938,0.618881
2,transmembrane,1,0.5,0.933181,0.419787,0.579078,0.940873,0.406411,0.567632
3,1.14,33,0.8,0.802817,0.422222,0.553398,0.824468,0.411141,0.548673
4,GO:0020037,33,0.8,0.976526,0.382353,0.549538,0.973404,0.353282,0.518414
5,GO:0007186,11,0.6,0.545932,0.497608,0.520651,0.546448,0.473934,0.507614


In [17]:
val_data = np.load(
    "Extraction/val_features/embeddings_layer_mean_128/embeddings_layer_mean_sparse_pca_128_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_layer_mean_128/embeddings_layer_mean_sparse_pca_128_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 128)
(25000, 128)
val f1: 0.13381
test f1: 0.12576
Validation pairs with F1 > 0.5: 10
Those also with test F1 > 0.5: 7
Survival rate: 0.7


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
2,krab,40,0.8,0.441860,0.730769,0.550725,0.488636,0.704918,0.577181
4,protein kinase,84,0.6,0.387998,0.934653,0.548359,0.414059,0.930594,0.573116
0,nudix hydrolase,109,0.8,0.870968,0.490909,0.627907,0.814815,0.431373,0.564103
1,cadherin,102,0.8,0.523256,0.762712,0.620690,0.438202,0.750000,0.553191
3,transmembrane,1,0.5,0.993708,0.379766,0.549522,0.996743,0.367158,0.536640
5,fad-binding fr-type,101,0.6,0.420455,0.711538,0.528571,0.400000,0.754717,0.522876
6,GO:0007186,11,0.6,0.549072,0.495215,0.520755,0.552486,0.473934,0.510204


In [18]:
val_data = np.load(
    "Extraction/val_features/embeddings_layer_mean_320/embeddings_layer_mean_sparse_pca_320_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_layer_mean_320/embeddings_layer_mean_sparse_pca_320_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 320)
(25000, 320)
val f1: 0.13843
test f1: 0.12988
Validation pairs with F1 > 0.5: 10
Those also with test F1 > 0.5: 8
Survival rate: 0.8


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
1,GO:0005506,33,0.8,0.928571,0.506083,0.655118,0.933673,0.491935,0.644366
2,heme,33,0.8,0.915179,0.488095,0.636646,0.903061,0.460938,0.610345
0,nudix hydrolase,109,0.8,1.000000,0.490909,0.658537,0.952381,0.392157,0.555556
3,cadherin,131,0.8,0.490566,0.881356,0.630303,0.411215,0.846154,0.553459
4,1.14,33,0.8,0.763393,0.422222,0.543720,0.790816,0.411141,0.541012
6,transmembrane,1,0.5,0.988139,0.372037,0.540554,0.990526,0.358416,0.526369
5,GO:0020037,33,0.8,0.928571,0.382353,0.541667,0.933673,0.353282,0.512605
8,GO:0007186,11,0.6,0.545213,0.490431,0.516373,0.546448,0.473934,0.507614


max

In [19]:
val_data = np.load(
    "Extraction/val_features/embeddings_max_8/embeddings_max_sparse_pca_8_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_max_8/embeddings_max_sparse_pca_8_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 8)
(25000, 8)
val f1: 0.07616
test f1: 0.07318
Validation pairs with F1 > 0.5: 2
Those also with test F1 > 0.5: 2
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,protein kinase,2,0.8,0.469663,0.620792,0.534755,0.469609,0.592694,0.524021
1,GO:0005525,6,0.6,0.384835,0.715426,0.500465,0.387023,0.723252,0.504227


In [20]:
val_data = np.load(
    "Extraction/val_features/embeddings_max_16/embeddings_max_sparse_pca_16_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_max_16/embeddings_max_sparse_pca_16_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 16)
(25000, 16)
val f1: 0.09653
test f1: 0.09228
Validation pairs with F1 > 0.5: 4
Those also with test F1 > 0.5: 4
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0003924,6,0.8,0.853306,0.753650,0.800388,0.867713,0.738550,0.797938
2,GO:0005525,6,0.8,0.929752,0.598404,0.728155,0.946188,0.601997,0.735833
1,abc transporter,15,0.8,0.840909,0.660714,0.740000,0.827586,0.592593,0.690647
3,GO:0003925,6,0.8,0.413223,0.913242,0.568990,0.374439,0.917582,0.531847


In [21]:
val_data = np.load(
    "Extraction/val_features/embeddings_max_32/embeddings_max_sparse_pca_32_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_max_32/embeddings_max_sparse_pca_32_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 32)
(25000, 32)
val f1: 0.1123
test f1: 0.10752
Validation pairs with F1 > 0.5: 5
Those also with test F1 > 0.5: 3
Survival rate: 0.6


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0005506,13,0.8,0.861925,0.501217,0.633846,0.789474,0.483871,0.600000
4,abc transporter,15,0.8,0.360902,0.857143,0.507937,0.495935,0.753086,0.598039
1,heme,13,0.8,0.845188,0.480952,0.613050,0.758772,0.450521,0.565359


In [22]:
val_data = np.load(
    "Extraction/val_features/embeddings_max_64/embeddings_max_sparse_pca_64_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_max_64/embeddings_max_sparse_pca_64_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 64)
(25000, 64)
val f1: 0.12719
test f1: 0.11928
Validation pairs with F1 > 0.5: 8
Those also with test F1 > 0.5: 7
Survival rate: 0.875


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0106310,56,0.8,0.638211,0.585821,0.610895,0.684783,0.593640,0.635962
2,protein kinase,56,0.8,0.696477,0.508911,0.588101,0.754076,0.506849,0.606226
3,GO:0004674,56,0.8,0.570461,0.565101,0.567768,0.631793,0.579800,0.604681
6,abc transporter,15,0.8,0.432099,0.625000,0.510949,0.547619,0.567901,0.557576
4,GO:0005506,13,0.8,0.714844,0.445255,0.548726,0.682203,0.432796,0.529605
1,cadherin,58,0.8,0.459459,0.864407,0.600000,0.380531,0.826923,0.521212
7,GO:0005634,8,0.5,0.387325,0.736849,0.507751,0.387799,0.733587,0.507380


In [23]:
val_data = np.load(
    "Extraction/val_features/embeddings_max_128/embeddings_max_sparse_pca_128_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_max_128/embeddings_max_sparse_pca_128_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 128)
(25000, 128)
val f1: 0.13825
test f1: 0.1288
Validation pairs with F1 > 0.5: 10
Those also with test F1 > 0.5: 9
Survival rate: 0.9


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,cadherin,58,0.8,0.924528,0.830508,0.875000,0.913043,0.807692,0.857143
1,protein kinase,79,0.6,0.618598,0.908911,0.736167,0.611912,0.891324,0.725651
2,GO:0106310,79,0.6,0.510782,0.942786,0.662587,0.497179,0.934040,0.648936
4,GO:0061630,65,0.6,0.605206,0.617257,0.611172,0.632911,0.618812,0.625782
3,GO:0003924,6,0.6,0.485801,0.874088,0.624511,0.493841,0.841603,0.622442
5,GO:0005525,6,0.6,0.534483,0.700798,0.606444,0.547592,0.697575,0.613551
6,GO:0004674,79,0.6,0.445418,0.887248,0.593091,0.447649,0.890274,0.595745
7,2.7,79,0.6,0.605121,0.550582,0.576565,0.605643,0.566569,0.585455
8,nudix hydrolase,61,0.8,0.435294,0.672727,0.528571,0.515152,0.666667,0.581197


In [24]:
val_data = np.load(
    "Extraction/val_features/embeddings_max_320/embeddings_max_sparse_pca_320_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_max_320/embeddings_max_sparse_pca_320_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 320)
(25000, 320)
val f1: 0.13005
test f1: 0.11851
Validation pairs with F1 > 0.5: 3
Those also with test F1 > 0.5: 3
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,protein kinase,36,0.6,0.663347,0.659406,0.661370,0.657774,0.652968,0.655362
1,GO:0106310,36,0.6,0.560757,0.700249,0.622788,0.553818,0.709069,0.621901
2,GO:0004674,36,0.6,0.494024,0.665772,0.567181,0.501380,0.679551,0.577025


mean

In [25]:
val_data = np.load(
    "Extraction/val_features/embeddings_mean_8/embeddings_mean_sparse_pca_8_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_mean_8/embeddings_mean_sparse_pca_8_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 8)
(25000, 8)
val f1: 0.06905
test f1: 0.06592
Validation pairs with F1 > 0.5: 1
Those also with test F1 > 0.5: 1
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0005634,3,0.6,0.477202,0.662906,0.55493,0.482461,0.661819,0.558084


In [26]:
val_data = np.load(
    "Extraction/val_features/embeddings_mean_16/embeddings_mean_sparse_pca_16_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_mean_16/embeddings_mean_sparse_pca_16_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 16)
(25000, 16)
val f1: 0.08342
test f1: 0.0784
Validation pairs with F1 > 0.5: 1
Those also with test F1 > 0.5: 1
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0005634,3,0.6,0.414945,0.739704,0.531653,0.421605,0.747451,0.539118


In [27]:
val_data = np.load(
    "Extraction/val_features/embeddings_mean_32/embeddings_mean_sparse_pca_32_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_mean_32/embeddings_mean_sparse_pca_32_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 32)
(25000, 32)
val f1: 0.09112
test f1: 0.08512
Validation pairs with F1 > 0.5: 2
Those also with test F1 > 0.5: 2
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0003925,17,0.8,0.517241,0.753425,0.613383,0.433447,0.697802,0.534737
1,GO:0005634,21,0.6,0.439873,0.603099,0.508714,0.448153,0.615060,0.518506


In [28]:
val_data = np.load(
    "Extraction/val_features/embeddings_mean_64/embeddings_mean_sparse_pca_64_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_mean_64/embeddings_mean_sparse_pca_64_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 64)
(25000, 64)
val f1: 0.10523
test f1: 0.10153
Validation pairs with F1 > 0.5: 1
Those also with test F1 > 0.5: 1
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0005634,3,0.5,0.374933,0.857551,0.52175,0.37852,0.864211,0.526455


In [29]:
val_data = np.load(
    "Extraction/val_features/embeddings_mean_128/embeddings_mean_sparse_pca_128_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_mean_128/embeddings_mean_sparse_pca_128_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 128)
(25000, 128)
val f1: 0.12169
test f1: 0.11427
Validation pairs with F1 > 0.5: 3
Those also with test F1 > 0.5: 3
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,nudix hydrolase,62,0.8,0.942857,0.600000,0.733333,1.000000,0.529412,0.692308
2,GO:0005634,3,0.5,0.376401,0.894522,0.529850,0.378903,0.897377,0.532828
1,6.2,104,0.8,0.458333,0.705128,0.555556,0.413793,0.705882,0.521739


In [30]:
val_data = np.load(
    "Extraction/val_features/embeddings_mean_320/embeddings_mean_sparse_pca_320_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_mean_320/embeddings_mean_sparse_pca_320_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 320)
(25000, 320)
val f1: 0.132
test f1: 0.12146
Validation pairs with F1 > 0.5: 4
Those also with test F1 > 0.5: 3
Survival rate: 0.75


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,nudix hydrolase,62,0.8,0.891892,0.600000,0.717391,0.710526,0.529412,0.606742
1,GO:0003925,256,0.8,0.550336,0.748858,0.634429,0.500000,0.675824,0.574766
2,GO:0005634,3,0.5,0.380184,0.887182,0.532273,0.381931,0.886639,0.533884


min

In [31]:
val_data = np.load(
    "Extraction/val_features/embeddings_min_8/embeddings_min_sparse_pca_8_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_min_8/embeddings_min_sparse_pca_8_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 8)
(25000, 8)
val f1: 0.05747
test f1: 0.05492
Validation pairs with F1 > 0.5: 0
Those also with test F1 > 0.5: 0
Survival rate: 0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1


In [32]:
val_data = np.load(
    "Extraction/val_features/embeddings_min_16/embeddings_min_sparse_pca_16_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_min_16/embeddings_min_sparse_pca_16_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 16)
(25000, 16)
val f1: 0.07898
test f1: 0.07499
Validation pairs with F1 > 0.5: 0
Those also with test F1 > 0.5: 0
Survival rate: 0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1


In [33]:
val_data = np.load(
    "Extraction/val_features/embeddings_min_32/embeddings_min_sparse_pca_32_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_min_32/embeddings_min_sparse_pca_32_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 32)
(25000, 32)
val f1: 0.09034
test f1: 0.08534
Validation pairs with F1 > 0.5: 1
Those also with test F1 > 0.5: 0
Survival rate: 0.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1


In [34]:
val_data = np.load(
    "Extraction/val_features/embeddings_min_64/embeddings_min_sparse_pca_64_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_min_64/embeddings_min_sparse_pca_64_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 64)
(25000, 64)
val f1: 0.10464
test f1: 0.0985
Validation pairs with F1 > 0.5: 4
Those also with test F1 > 0.5: 3
Survival rate: 0.75


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,protein kinase,60,0.6,0.473154,0.698020,0.564000,0.495920,0.721461,0.587798
2,GO:0106310,60,0.6,0.398658,0.738806,0.517873,0.424357,0.796231,0.553645
1,GO:0005634,58,0.5,0.402219,0.744189,0.522199,0.400452,0.745820,0.521107


In [35]:
val_data = np.load(
    "Extraction/val_features/embeddings_min_128/embeddings_min_sparse_pca_128_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_min_128/embeddings_min_sparse_pca_128_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 128)
(25000, 128)
val f1: 0.11455
test f1: 0.10542
Validation pairs with F1 > 0.5: 1
Those also with test F1 > 0.5: 0
Survival rate: 0.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1


In [36]:
val_data = np.load(
    "Extraction/val_features/embeddings_min_320/embeddings_min_sparse_pca_320_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_min_320/embeddings_min_sparse_pca_320_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)
mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)
A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 320)
(25000, 320)
val f1: 0.1265
test f1: 0.11559
Validation pairs with F1 > 0.5: 2
Those also with test F1 > 0.5: 2
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,vwfa,180,0.8,0.468354,0.649123,0.544118,0.536232,0.637931,0.582677
1,GO:0005634,30,0.5,0.393098,0.689004,0.500593,0.393832,0.692538,0.502119
